In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, paths
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "wheat"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
rescaled_pregnancy_root = paths.source_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT, data_source)
rescaled_child_root = paths.source_root(paths.RESCALED_CHILD_RESULTS_ROOT, data_source)
anemia_root = paths.source_root(paths.NON_PREGNANT_ANEMIA_RESULTS_ROOT, data_source)
print(f"{data_source}:\n  {rescaled_pregnancy_root}\n  {rescaled_child_root}\n  {anemia_root}")

model2.6.4:
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/rescaled_pregnancy
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/rescaled_child
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/non_pregnant_anemia


In [5]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [6]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylls.parquet"
if has_simulations:
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,141,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,141,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,141,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,141,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,zero,0,129,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,zero,0,129,0.0
539997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,zero,0,129,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,0,129,0.0


In [7]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [8]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [9]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  430866.935390
                                  2                  256605.660716
                                  3                  305054.203059
                                  4                  295526.150075
                                  5                  177030.945234
intervention  maternal_disorders  1                  429679.733431
                                  2                  256462.414465
                                  3                  304891.538496
                                  4                  293981.698565
                                  5                  176396.972207
zero          maternal_disorders  1                  431028.670089
                                  2                  256605.660716
                                  3                  305054.203059
                                  4                  295526.150075
            

In [10]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,141,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,141,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,141,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,141,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,0,129,0.0
1889996,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,0,129,0.0
1889997,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,0,129,0.0
1889998,ylds,cause,all_causes,all_causes,95_plus,severe,5,zero,0,129,0.0


In [11]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [12]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  107044.305128
                                  2                   82910.093868
                                  3                   66598.460415
                                  4                   56913.393407
                                  5                   42769.682856
              maternal_disorders  1                   75541.661262
                                  2                   40030.918702
                                  3                   46564.377845
                                  4                   46276.680262
                                  5                   32080.693949
intervention  anemia              1                  106044.720015
                                  2                   82115.582208
                                  3                   65892.153950
                                  4                   56291.766818
            

In [13]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  107044.305128
                                  2                   82910.093868
                                  3                   66598.460415
                                  4                   56913.393407
                                  5                   42769.682856
              maternal_disorders  1                  506408.596652
                                  2                  296636.579417
                                  3                  351618.580904
                                  4                  341802.830337
                                  5                  209111.639183
intervention  anemia              1                  106044.720015
                                  2                   82115.582208
                                  3                   65892.153950
                                  4                   56291.766818
            

In [14]:
ylds_path = f"{rescaled_child_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [15]:
path = f"{rescaled_child_root}/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # Combos with no iron pathway (e.g. ethiopia/salt, nigeria/salt) have no
    # Vivarium sims, so the corresponding DALYs are 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"{rescaled_child_root}/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,41810.570637
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,33142.820122
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,28597.306682
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,22099.062794
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,22744.377339
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,3560.441002
23996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,2105.100203
23997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,3551.444476
23998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,2728.249651


In [16]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.795077e+07
                      2                  1.483410e+07
                      3                  1.314924e+07
                      4                  1.253438e+07
                      5                  1.185806e+07
intervention  lbwsg   1                  1.794383e+07
                      2                  1.482694e+07
                      3                  1.314620e+07
                      4                  1.253005e+07
                      5                  1.185199e+07
zero          lbwsg   1                  1.795185e+07
                      2                  1.483431e+07
                      3                  1.314946e+07
                      4                  1.253503e+07
                      5                  1.185806e+07
Name: value, dtype: float64

In [17]:
path = f"{anemia_root}/{vehicle}/{location}/ylds.parquet"
if has_anemia_model:
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{anemia_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1189.250795,zero
1,Female,0.0,0.019178,2,903.589776,zero
2,Female,0.0,0.019178,3,800.748345,zero
3,Female,0.0,0.019178,4,665.963014,zero
4,Female,0.0,0.019178,5,466.292372,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,244.262340,intervention
746,Male,95.0,125.000000,2,237.896304,intervention
747,Male,95.0,125.000000,3,243.685765,intervention
748,Male,95.0,125.000000,4,236.417157,intervention


In [18]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.959952e+06
                      2                  1.917810e+06
                      3                  1.950357e+06
                      4                  1.851170e+06
                      5                  1.651247e+06
intervention  anemia  1                  1.940979e+06
                      2                  1.898984e+06
                      3                  1.930479e+06
                      4                  1.831052e+06
                      5                  1.630387e+06
zero          anemia  1                  1.961121e+06
                      2                  1.918970e+06
                      3                  1.951582e+06
                      4                  1.852410e+06
                      5                  1.652532e+06
Name: value, dtype: float64

In [19]:
scenarios[1]

'zero'

In [20]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

np.float64(-6335.69923186861)

In [21]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  666235.672613
                      2                  544530.503858
                      3                  499396.349739
                      4                  426103.063573
                      5                  327548.172982
intervention  anemia  1                  660244.638123
                      2                  539672.098098
                      3                  494948.364520
                      4                  421990.131273
                      5                  324023.794694
zero          anemia  1                  666604.883452
                      2                  544829.913930
                      3                  499670.466730
                      4                  426356.532182
                      5                  327765.370641
Name: value, dtype: float64

In [22]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

np.float64(-1413.404170430731)

In [23]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.739310e+06
                      2                  4.299964e+06
                      3                  4.218340e+06
                      4                  3.870370e+06
                      5                  3.378676e+06
intervention  anemia  1                  4.693596e+06
                      2                  4.258101e+06
                      3                  4.175935e+06
                      4                  3.828794e+06
                      5                  3.336621e+06
zero          anemia  1                  4.742127e+06
                      2                  4.302544e+06
                      3                  4.220954e+06
                      4                  3.872932e+06
                      5                  3.381268e+06
Name: value, dtype: float64

In [24]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if has_ntd_model:
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  293872.767582
                      2                  259614.669023
                      3                  233065.195473
                      4                  219760.493462
                      5                  187191.840608
baseline      ntd     1                  293733.701862
                      2                  259483.070661
                      3                  232954.812359
                      4                  219664.907598
                      5                  187079.380834
intervention  ntd     1                  291969.887485
                      2                  257814.877305
                      3                  231554.797545
                      4                  218451.810089
                      5                  185655.959250
Name: value, dtype: float64

In [25]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.846354e+06
                                  2                  4.382874e+06
                                  3                  4.284939e+06
                                  4                  3.927283e+06
                                  5                  3.421446e+06
              lbwsg               1                  1.795077e+07
                                  2                  1.483410e+07
                                  3                  1.314924e+07
                                  4                  1.253438e+07
                                  5                  1.185806e+07
              maternal_disorders  1                  5.064086e+05
                                  2                  2.966366e+05
                                  3                  3.516186e+05
                                  4                  3.418028e+05
                          

In [26]:
import pathlib

In [27]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)